In [1]:
from itertools import product
from loguru import logger
from pathlib import Path
import pandas as pd
import numpy as np
import time
from tqdm import tqdm
import time
from dataclasses import asdict

import combined_cooler
from solhycool_modeling import EnvironmentVariables, OperationPoint, ModelInputsRange
from solhycool_optimization import DecisionVariables, ValuesDecisionVariables

cc_model = combined_cooler.initialize()

%load_ext autoreload
%autoreload 2

data_path: Path = Path("../../data")
env_path: Path = data_path / "datasets/environment_data_20220101_20241231.h5"


Authorization required, but no authorization protocol specified



# One step evaluation

In [3]:
# Load environment into EnvironmentVariables for the episode
df_env = pd.read_hdf(env_path)

selected_date_str: str = "20220103" 
df_day = df_env.loc[selected_date_str]


In [16]:
# Generate decision variable arrays
dv_values: ValuesDecisionVariables = ValuesDecisionVariables().generate_arrays()

# Prepare environment variables
ev = EnvironmentVariables.from_series(df_day.iloc[0]).contrain_to_model()
ev_m = ev.to_matlab()

# Initialize results storage
dv_list: list[DecisionVariables] = []
consumption_list: list[list[float], list[float]] = [[], []]

# Compute total number of evaluations
total_num_evals = np.prod([len(value) for value in asdict(dv_values).values()])

start_time = time.time()
with tqdm(total=total_num_evals, desc="Evaluating decision variables") as pbar:
    for qc_val in dv_values.qc:
        for rp_val in dv_values.Rp:
            for rs_val in dv_values.Rs:
                for wdc_val in dv_values.wdc:
                    dv = DecisionVariables(qc=qc_val, Rp=rp_val, Rs=rs_val, wdc=wdc_val).to_matlab()
                    Ce_kWe, Cw_lh, d, valid = cc_model.evaluate_operation(
                        ev_m.Tamb, ev_m.HR, ev_m.mv, dv.qc, dv.Rp, dv.Rs, dv.wdc, ev_m.Tv, nargout=4
                    )
                    
                    if valid:
                        dv_list.append(DecisionVariables(qc=d["qc"], Rp=d["Rp"], Rs=d["Rs"], wdc=d["wdc"], wwct=d["wwct"]))
                        consumption_list[0].append(Cw_lh)
                        consumption_list[1].append(Ce_kWe)

                    # Update progress bar and dynamically update candidate count
                    pbar.update(1)
                    pbar.set_postfix(valid_candidates=len(dv_list))

logger.info(f"Generated {len(dv_list)} decision variable candidates to evaluate in {time.time() - start_time:.1f} seconds")


Evaluating decision variables:   0%|          | 0/10000 [00:00<?, ?it/s]

Evaluating decision variables: 100%|██████████| 10000/10000 [05:22<00:00, 30.97it/s, valid_candidates=2035]
2025-03-27 18:41:13.808 | INFO     | __main__:<module>:35 - Generated 2035 decision variable candidates to evaluate in 322.9 seconds


In [19]:
# Save the decision variables to a CSV file
pd.DataFrame( [asdict(dv) for dv in dv_list] ).to_csv("dv_candidates.csv", index=False)

# Save the consumption data to a CSV file
np.array(consumption_list).T.tofile("consumption_candidates.csv", sep=",", format="%f")


In [ ]:
# Load the decision variables from the CSV file
dv_list = [ DecisionVariables(**dv) for dv in pd.read_csv("dv_candidates.csv").to_dict(orient="records")]
display(dv_list[0])

# Load the consumption data from the CSV file
consumption_list = np.fromfile("consumption_candidates.csv", sep=",", dtype=float).reshape(-1, 2).tolist()


DecisionVariables(qc=9.428477777777776, Rp=0.2222222222222222, Rs=0.4444444444444444, wdc=59.988888888888894, wwct=37.704242450647214)

In [5]:
dv = dv_list[0]
[ev.Tamb, ev.HR, ev.mv, dv.qc, dv.Rp, dv.Rs, dv.wdc, ev.Tv]


[9.06,
 np.float64(73.0),
 np.float64(297.7741653659358),
 9.428477777777777,
 0.2222222222222222,
 0.4444444444444444,
 59.988888888888894,
 np.float64(35.0)]

In [ ]:
from solhycool_optimization.utils import pareto_front_indices

consumption_array = np.array(consumption_list).transpose()
idxs = pareto_front_indices(consumption_array, objective="minimize")

logger.info(f"Pareto front indices: {idxs}")

# Generate operation points for the Pareto front
ops = [
    OperationPoint.from_multiple_sources(
        dict_src=cc_model.evaluate_operation(ev_m.Tamb, ev_m.HR, ev_m.mv, dv.qc, dv.Rp, dv.Rs, dv.wdc, ev_m.Tv, nargout=3)[2],
        env_vars=ev,
        time=df_day.index[0],
    )
    for dv in [dv_list[i] for i in idxs]
]


2025-03-27 18:43:00.112 | INFO     | __main__:<module>:6 - Pareto front indices: [1174 1421 1680 1182 1189 1420 1181 1188 1176    0  241  535    5  250
  257  298  320  110  382  141]


In [38]:
# Visualize pareto front
from solhycool_optimization.visualization.pareto import plot_pareto_front

plot_pareto_front(ops_list=[ops], additional_pts=consumption_array, full_legend=True)


# Multiple steps evaluation

In [2]:
# Obtener frente para todos los pasos en paralelo

import time
import numpy as np
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path
from dataclasses import asdict

import combined_cooler
from solhycool_modeling import EnvironmentVariables
from solhycool_optimization import DecisionVariables, ValuesDecisionVariables
from solhycool_optimization.utils import pareto_front_indices

def evaluate_decision_variables(step_idx: int, ds_env: pd.Series, dv_values: ValuesDecisionVariables, total_num_evals: int) -> tuple[list[DecisionVariables], list[list[float], list[float]]]:
    """Evaluates decision variables for a given step."""
    
    cc_model = combined_cooler.initialize()
    
    # Prepare environment variables
    ev = EnvironmentVariables.from_series(ds_env).constrain_to_model()
    ev_m = ev.to_matlab()
    
    dv_list = []
    consumption_list = [[], []]
    
    with tqdm(total=total_num_evals, desc=f"Step {step_idx}", position=step_idx, leave=False) as pbar:
        for qc_val in dv_values.qc:
            for rp_val in dv_values.Rp:
                for rs_val in dv_values.Rs:
                    for wdc_val in dv_values.wdc:
                        dv = DecisionVariables(qc=qc_val, Rp=rp_val, Rs=rs_val, wdc=wdc_val).to_matlab()
                        Ce_kWe, Cw_lh, d, valid = cc_model.evaluate_operation(
                            ev_m.Tamb, ev_m.HR, ev_m.mv, dv.qc, dv.Rp, dv.Rs, dv.wdc, ev_m.Tv, nargout=4
                        )
                        if valid:
                            dv_list.append(DecisionVariables(qc=d["qc"], Rp=d["Rp"], Rs=d["Rs"], wdc=d["wdc"], wwct=d["wwct"]))
                            consumption_list[0].append(Cw_lh)
                            consumption_list[1].append(Ce_kWe)
                        
                        pbar.update(1)
                        pbar.set_postfix(valid_candidates=len(dv_list))
    
    return dv_list, consumption_list


In [ ]:
n_parallel_evals = 10
selected_date_str: str = "20220103" 

# Load environment into EnvironmentVariables for the episode
df_env = pd.read_hdf(env_path)
df_day = df_env.loc[selected_date_str]

# Generate decision variable arrays
dv_values: ValuesDecisionVariables = ValuesDecisionVariables().generate_arrays()
# Compute total number of evaluations
total_num_evals = np.prod([len(value) for value in asdict(dv_values).values()])

results = []
start_time = time.time()

with ProcessPoolExecutor(max_workers=n_parallel_evals) as executor:
    futures = {executor.submit(evaluate_decision_variables, step_idx, ds, dv_values, total_num_evals): step_idx for step_idx, (dt, ds) in enumerate(df_day.iterrows())}
    for future in as_completed(futures):
        step_idx = futures[future]
        
        dv_lists, consumption_list = future.result()
        consumption_array = np.array(consumption_list).transpose()
        idxs = pareto_front_indices(consumption_array, objective="minimize")

        logger.info(f"Pareto front indices: {idxs}")

        # Generate operation points for the Pareto front
        ops = [
            OperationPoint.from_multiple_sources(
                dict_src=cc_model.evaluate_operation(ev_m.Tamb, ev_m.HR, ev_m.mv, dv.qc, dv.Rp, dv.Rs, dv.wdc, ev_m.Tv, nargout=3)[2],
                env_vars=ev,
                time=df_day.index[step_idx],
            )
            for dv in [dv_list[i] for i in idxs]
        ]
        df_paretos = [pd.DataFrame(asdict(op)) for op in ops]
        results.append((step_idx, df_paretos, consumption_array))
            

# Save results to HDF5
output_path = Path(".") / "results" / "pareto_fronts.h5"
with pd.HDFStore(output_path, mode='w') as store:
    for step_idx, df_paretos, _ in results:
        store.put(f"step_{step_idx}", pd.concat(df_paretos))

print(f"Completed evaluation in {time.time() - start_time:.1f} seconds")
